<a href="https://colab.research.google.com/github/MayerT1/PIPECAST/blob/flood_mapping/flood_mapping/notebooks/alaska_pipecast_round2_perturbations.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Introduction

In this module, we will take a sample image from ICEYE and run several different perturbations of the traditional HYDRAFloods workflows

1. Terrain Correction
  - We will use the USGS 3DEP Digital Elevation Model and will run the terrain correction at the native scale of this map, which is 10 meters
  
2. Speckle Filter

  We will use the p_median filter, and will test windows of different sizes: **window of 3, 5, 7, 13, and 19 pixels**

3. Edge Otsu Algorithm

  - connected_pixel parameter. We will test the following values for this parametetr: **50, 100, 200, 300, 350**
  - initial_threshold parameter. This is determined using a sampling method. We will test the following parameters: **1e2, 1e3, 1e4, 1e5, 1e6**
  - thresh_no_data parameter. This will be determined based on the histogram. We will use **mean - 2 standard deviations, mean - 1 standard deviations, mean + 1 standard deviations, mean + 2 standard deviations**
  - We will use the native scale of the image to run the Edge Otsu algorithm


  After we run all combinations of the 5 values for each four parameters, we will pick the 3 best combinations, then we will use those to test the Canny Edge Detector portion of the Edge Otsu algorithm.

  - canny_threshold
  - canny_sigma
  - canny_lt

# Part 0: Housekeeping and obtaining ICEYE image

In [ ]:
my_gee_folder = 'users/mickymags/watchduty/'

In [ ]:
!pip install ipyleaflet==0.18.2 geemap hydrafloods     # Install hydrafloods and its relevant dependencies
!pip install geemap

In [ ]:
from hydrafloods import corrections
import hydrafloods as hf
import ee
import geemap
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
ee.Authenticate()
ee.Initialize(project = 'servir-sco-assets')

In [ ]:
aoi = ee.FeatureCollection('users/mickymags/watchduty/ak_aoi1')

aoi_geom = aoi.geometry()

# Get the coordinates of the center of the AOI for mapping purposes
aoi_centroid = aoi.geometry().centroid()             # Get the center of the AOI
lon = aoi_centroid.coordinates().get(0).getInfo()    # Extract the longitude from the centroid
lat = aoi_centroid.coordinates().get(1).getInfo()    # Extract the latitude from the centroid

In [ ]:
iceye = hf.Dataset(
    region = aoi_geom,               #my_geom
    start_time = '2025-10-13',
    end_time = '2025-10-15',
    asset_id = 'users/mickymags/watchduty/alaska_iceye_v2'
)

In [ ]:
iceye_first = iceye.collection.first()

In [ ]:
#iceye_med = iceye.collection.median()

In [ ]:
#iceye_med = iceye_med.rename('VV', 'angle').clip(aoi_geom)
iceye_first = iceye_first.rename('VV', 'angle').clip(aoi_geom)

In [ ]:
#iceye_med.bandNames().getInfo()
iceye_first.bandNames().getInfo()

In [ ]:
iceye_vp = {
    'bands': 'VV', #'b1',
    'min': 0,
    'max': 2000
}

# Step 1: Pre-Processing

In [ ]:
from hydrafloods import corrections

### Step 1a: Terrain Correction

In [ ]:
elv = ee.ImageCollection("USGS/3DEP/10m_collection").filterBounds(aoi_geom).mosaic().clip(aoi_geom)
#elv_filt = ee.ImageCollection("USGS/3DEP/10m_collection").filterBounds(aoi_geom)

#elv = elv_filt.mosaic().clip(aoi_geom)

#elv_scale = elv_filt.first().projection().nominalScale().getInfo()


In [ ]:
elv.projection().nominalScale().getInfo()

In [ ]:
#elv_proj.projection().getInfo()

In [ ]:
#print(elv_scale)

In [ ]:
#elv_scale = ee.Number(1000)

In [ ]:
elv_proj = elv.reproject(crs = iceye_first.projection(), scale = 10)

In [ ]:
ic_terrain = corrections.slope_correction(iceye_first, elevation=elv_proj, scale = 10) #elevation=elv_proj

In [ ]:
elv_params = {
    'min': 0,
    'max': 3
}

In [ ]:
Map = geemap.Map(center = (lat, lon), zoom = 11)
Map.addLayer(aoi_geom, {}, 'Area of Interest')
Map.addLayer(iceye_first, iceye_vp, 'ICEYE Reg')
Map.addLayer(ic_terrain, iceye_vp, 'Terrain corrected')
Map.addLayer(elv_proj, elv_params, 'Elevation')

Map.addLayerControl()
Map

### Step 1b: Speckle Filter

In [ ]:
ic_pmedian_3 = hf.p_median(ic_terrain, window=3)
ic_pmedian_5 = hf.p_median(ic_terrain, window=5)
ic_pmedian_7 = hf.p_median(ic_terrain, window=7)
ic_pmedian_9 = hf.p_median(ic_terrain, window=9)
ic_pmedian_11 = hf.p_median(ic_terrain, window=11)

In [ ]:
Map = geemap.Map(center = (lat, lon), zoom = 11)
Map.addLayer(aoi_geom, {}, 'Area of Interest')
Map.addLayer(iceye_first, iceye_vp, 'ICEYE Reg')
Map.addLayer(ic_pmedian_3, iceye_vp, 'ICEYE3')
Map.addLayer(ic_pmedian_5, iceye_vp, 'ICEYE5')
Map.addLayer(ic_pmedian_7, iceye_vp, 'ICEYE7')
Map.addLayer(ic_pmedian_9, iceye_vp, 'ICEYE9')
Map.addLayer(ic_pmedian_11, iceye_vp, 'ICEYE11')

Map.addLayerControl()
Map

# Step 2: Determine Initial threshold and Thresh No Data

### Step 2a: Determining the Initial Threshold

In [ ]:
iceye_scale = iceye_first.projection().nominalScale().getInfo()

In [ ]:
iceye_scale

In [ ]:
def init_thresh_determiner(image, roi, numpix, my_scale):
  image_sampled = image.sample(roi, numPixels=numpix, scale=my_scale).toList(numpix).getInfo()

  #features = image_sampled['features']
  vals = []
  for j in range(int(numpix)):#(len(features)):
    #print(j)
    if j < numpix - 0.02*numpix:                    # Flag to stop a weird bug from occuring where earthengine doesn't count all the way to numpix
      feat_of_int = image_sampled[j]
      value = feat_of_int['properties']['VV']
      vals.append(value)

  np_vals = np.array(vals)
  my_mean = np_vals.mean()
  return my_mean

In [ ]:
# Terrain corrected ICEYE image speckle filtered with a window of 3, sampled for 100 pixels
ic_pmedian_3_1e2_threshInit = init_thresh_determiner(ic_pmedian_3, aoi, 1e2, iceye_scale)
ic_pmedian_3_1e3_threshInit = init_thresh_determiner(ic_pmedian_3, aoi, 1e3, iceye_scale)
ic_pmedian_3_1e4_threshInit = init_thresh_determiner(ic_pmedian_3, aoi, 1e4, iceye_scale)
ic_pmedian_3_1e5_threshInit = init_thresh_determiner(ic_pmedian_3, aoi, 1e5, iceye_scale)
ic_pmedian_3_1e6_threshInit = init_thresh_determiner(ic_pmedian_3, aoi, 1e6, iceye_scale)

# Terrain corrected ICEYE image speckle filtered with a window of 5, sampled for 100 pixels
ic_pmedian_5_1e2_threshInit = init_thresh_determiner(ic_pmedian_5, aoi, 1e2, iceye_scale)
ic_pmedian_5_1e3_threshInit = init_thresh_determiner(ic_pmedian_5, aoi, 1e3, iceye_scale)
ic_pmedian_5_1e4_threshInit = init_thresh_determiner(ic_pmedian_5, aoi, 1e4, iceye_scale)
ic_pmedian_5_1e5_threshInit = init_thresh_determiner(ic_pmedian_5, aoi, 1e5, iceye_scale)
ic_pmedian_5_1e6_threshInit = init_thresh_determiner(ic_pmedian_5, aoi, 1e6, iceye_scale)

# Terrain corrected ICEYE image speckle filtered with a window of 7, sampled for 100 pixels
ic_pmedian_7_1e2_threshInit = init_thresh_determiner(ic_pmedian_7, aoi, 1e2, iceye_scale)
ic_pmedian_7_1e3_threshInit = init_thresh_determiner(ic_pmedian_7, aoi, 1e3, iceye_scale)
ic_pmedian_7_1e4_threshInit = init_thresh_determiner(ic_pmedian_7, aoi, 1e4, iceye_scale)
ic_pmedian_7_1e5_threshInit = init_thresh_determiner(ic_pmedian_7, aoi, 1e5, iceye_scale)
ic_pmedian_7_1e6_threshInit = init_thresh_determiner(ic_pmedian_7, aoi, 1e6, iceye_scale)

# Terrain corrected ICEYE image speckle filtered with a window of 9, sampled for 100 pixels
ic_pmedian_9_1e2_threshInit = init_thresh_determiner(ic_pmedian_9, aoi, 1e2, iceye_scale)
ic_pmedian_9_1e3_threshInit = init_thresh_determiner(ic_pmedian_9, aoi, 1e3, iceye_scale)
ic_pmedian_9_1e4_threshInit = init_thresh_determiner(ic_pmedian_9, aoi, 1e4, iceye_scale)
ic_pmedian_9_1e5_threshInit = init_thresh_determiner(ic_pmedian_9, aoi, 1e5, iceye_scale)
ic_pmedian_9_1e6_threshInit = init_thresh_determiner(ic_pmedian_9, aoi, 1e6, iceye_scale)

# Terrain corrected ICEYE image speckle filtered with a window of 11, sampled for 100 pixels
ic_pmedian_11_1e2_threshInit = init_thresh_determiner(ic_pmedian_11, aoi, 1e2, iceye_scale)
ic_pmedian_11_1e3_threshInit = init_thresh_determiner(ic_pmedian_11, aoi, 1e3, iceye_scale)
ic_pmedian_11_1e4_threshInit = init_thresh_determiner(ic_pmedian_11, aoi, 1e4, iceye_scale)
ic_pmedian_11_1e5_threshInit = init_thresh_determiner(ic_pmedian_11, aoi, 1e5, iceye_scale)
ic_pmedian_11_1e6_threshInit = init_thresh_determiner(ic_pmedian_11, aoi, 1e6, iceye_scale)

print('Pmedian window of 3 pixels')
print('100 samples:',ic_pmedian_3_1e2_threshInit)
print('1,000 samples:',ic_pmedian_3_1e3_threshInit)
print('10,000 samples:',ic_pmedian_3_1e4_threshInit)
print('100,000 samples:',ic_pmedian_3_1e5_threshInit)
print('1,000,000 samples:',ic_pmedian_3_1e6_threshInit)

print('--------------------------------------------------')
print('Pmedian window of 5 pixels')
print('100 samples:', ic_pmedian_5_1e2_threshInit)
print('1,000 samples:', ic_pmedian_5_1e3_threshInit)
print('10,000 samples:', ic_pmedian_5_1e4_threshInit)
print('100,000 samples:', ic_pmedian_5_1e5_threshInit)
print('1,000,000 samples:', ic_pmedian_5_1e6_threshInit)

print('--------------------------------------------------')
print('Pmedian window of 7 pixels')
print('100 samples:', ic_pmedian_7_1e2_threshInit)
print('1,000 samples:', ic_pmedian_7_1e3_threshInit)
print('10,000 samples:', ic_pmedian_7_1e4_threshInit)
print('100,000 samples:', ic_pmedian_7_1e5_threshInit)
print('1,000,000 samples:', ic_pmedian_7_1e6_threshInit)

print('--------------------------------------------------')
print('Pmedian window of 9 pixels')
print('100 samples:', ic_pmedian_9_1e2_threshInit)
print('1,000 samples:', ic_pmedian_9_1e3_threshInit)
print('10,000 samples:', ic_pmedian_9_1e4_threshInit)
print('100,000 samples:', ic_pmedian_9_1e5_threshInit)
print('1,000,000 samples:', ic_pmedian_9_1e6_threshInit)

print('----------------------------------------------------')
print('Pmedian window of 11 pixels')
print('100 samples:', ic_pmedian_11_1e2_threshInit)
print('1,000 samples:', ic_pmedian_11_1e3_threshInit)
print('10,000 samples:', ic_pmedian_11_1e4_threshInit)
print('100,000 samples:', ic_pmedian_11_1e5_threshInit)
print('1,000,000 samples:', ic_pmedian_11_1e6_threshInit)

### Visualize Initial Thresholds on Histograms

In [ ]:
#plot_histograms(ic_pmedian_3, ['VV'], 'Pmedian3', aoi_geom, 0, 1000, scale = iceye_scale, n_bins = 1024)
#plt.axvline()

In [ ]:
#plot_histograms(ic_pmedian_5, ['VV'], 'Pmedian5', aoi_geom, 0, 1000, scale = iceye_scale, n_bins = 1024)

In [ ]:
# Create a histogram for each band
def plot_histograms_w_thresholds(image, bands,  aoi_descriptor, region, xmin, xmax, threshold_list, final_thresh, scale=3.46325, n_bins=4096):
    """Plot histograms for each band of an ee.Image."""
    fig, axes = plt.subplots(len(bands), 1, figsize=(8, 4*len(bands)))

    if len(bands) == 1:
        axes = [axes]

    for i, band in enumerate(bands):
        # Reduce region to histogram
        hist_dict = image.select(band).reduceRegion(
            reducer=ee.Reducer.histogram(maxBuckets=n_bins), #maxBuckets=n_bins),
            geometry=region,
            scale=scale,

            bestEffort=True
        ).get(band).getInfo()

        if hist_dict is not None:
            counts = hist_dict['histogram']
            bucketMeans = hist_dict['bucketMeans']

            axes[i].bar(bucketMeans, counts, width=(bucketMeans[1]-bucketMeans[0]))
            axes[i].set_title(f"Histogram of {band} for {aoi_descriptor}")
            axes[i].set_xlabel("Pixel values")
            axes[i].set_ylabel("Frequency")
            axes[i].set_xlim(xmin, xmax) #0, 250
        else:
            axes[i].set_title(f"{band}: No data")

    for j in threshold_list:
      thresh_value = j[0]
      thresh_label = j[1]
      thresh_color = j[2]

      #print(thresh_value)
      #print(thresh_label)
      #print(thresh_color)
      plt.axvline(thresh_value, color=thresh_color, label=thresh_label)

    plt.axvline(final_thresh, color='#000000', label='Final Threshold')


    plt.legend(loc='upper right')
    plt.grid()
    plt.tight_layout()
    plt.show()

**Histogram Visualization for a Window of 3 pixels**

In [ ]:
win3_dict = [[ic_pmedian_3_1e2_threshInit, '100 samples', '#FF0000'],
            [ic_pmedian_3_1e3_threshInit, '1,000 samples', '#00FF00'],
            [ic_pmedian_3_1e4_threshInit, '10,000 samples', '#FFa500'],
            [ic_pmedian_3_1e5_threshInit, '100,000 samples', '#FFFF00'],
             [ic_pmedian_3_1e6_threshInit, '1,000,000 samples', '#800080']]

In [ ]:
plot_histograms_w_thresholds(ic_pmedian_3, ['VV'], 'Pmedian Filter, Window of 3 pixels', aoi_geom, 0, 1200, win3_dict, 614, scale = iceye_scale, n_bins = 1024)

Let's take a closer look at this

In [ ]:
plot_histograms_w_thresholds(ic_pmedian_3, ['VV'], 'Pmedian Filter, Window of 3 pixels', aoi_geom, 560, 600, win3_dict, 614, scale = iceye_scale, n_bins = 1024)

**Histogram Visualization for a Window of 5 pixels**

In [ ]:
win5_dict = [[ic_pmedian_5_1e2_threshInit, '100 samples', '#FF0000'],
            [ic_pmedian_5_1e3_threshInit, '1,000 samples', '#00FF00'],
            [ic_pmedian_5_1e4_threshInit, '10,000 samples', '#FFa500'],
            [ic_pmedian_5_1e5_threshInit, '100,000 samples', '#FFFF00'],
             [ic_pmedian_5_1e6_threshInit, '1,000,000 samples', '#800080']]

In [ ]:
plot_histograms_w_thresholds(ic_pmedian_5, ['VV'], 'Pmedian Filter, Window of 5 pixels', aoi_geom, 0, 1200, win5_dict, 582, scale = iceye_scale, n_bins = 1024)

**Histogram Visualization for a Window of 7 pixels**

In [ ]:
win7_dict = [[ic_pmedian_7_1e2_threshInit, '100 samples', '#FF0000'],
            [ic_pmedian_7_1e3_threshInit, '1,000 samples', '#00FF00'],
            [ic_pmedian_7_1e4_threshInit, '10,000 samples', '#FFa500'],
            [ic_pmedian_7_1e5_threshInit, '100,000 samples', '#FFFF00'],
             [ic_pmedian_7_1e6_threshInit, '1,000,000 samples', '#800080']]

In [ ]:
plot_histograms_w_thresholds(ic_pmedian_7, ['VV'], 'Pmedian Filter, Window of 7 pixels', aoi_geom, 0, 1200, win7_dict, 578, scale = iceye_scale, n_bins = 1024)

**Histogram Visualization for a Window of 9 pixels**

In [ ]:
win9_dict = [[ic_pmedian_9_1e2_threshInit, '100 samples', '#FF0000'],
            [ic_pmedian_9_1e3_threshInit, '1,000 samples', '#00FF00'],
            [ic_pmedian_9_1e4_threshInit, '10,000 samples', '#FFa500'],
            [ic_pmedian_9_1e5_threshInit, '100,000 samples', '#FFFF00'],
             [ic_pmedian_9_1e6_threshInit, '1,000,000 samples', '#800080']]

In [ ]:
plot_histograms_w_thresholds(ic_pmedian_9, ['VV'], 'Pmedian Filter, Window of 9 pixels', aoi_geom, 0, 1200, win9_dict, 569, scale = iceye_scale, n_bins = 1024)

**Histogram Visualization for a Window of 11 pixels**

In [ ]:
win11_dict = [[ic_pmedian_11_1e2_threshInit, '100 samples', '#FF0000'],
            [ic_pmedian_11_1e3_threshInit, '1,000 samples', '#00FF00'],
            [ic_pmedian_11_1e4_threshInit, '10,000 samples', '#FFa500'],
            [ic_pmedian_11_1e5_threshInit, '100,000 samples', '#FFFF00'],
             [ic_pmedian_11_1e6_threshInit, '1,000,000 samples', '#800080']]

In [ ]:
plot_histograms_w_thresholds(ic_pmedian_11, ['VV'], 'Pmedian Filter, Window of 11 pixels', aoi_geom, 0, 1200, win9_dict, 561, scale = iceye_scale, n_bins = 256)

# Step 2b: Determine thresh_no_data for each speckle window value

For each of the above histograms, we want to find the following values

* the mean
* the value 2 standard deviations below the mean
* the value 1 standard deviations below the mean
* the value 1 standard deviation above the mean
* the value 2 standard deviations above the mean

In [ ]:
def value_obtainer(image, roi, numpix, my_scale):
  image_sampled = image.sample(roi, numPixels=numpix, scale=my_scale).toList(numpix).getInfo()

  #features = image_sampled['features']
  vals = []
  for j in range(int(numpix)):#(len(features)):
    #print(j)
    if j < numpix - 0.02*numpix:                    # Flag to stop a weird bug from occuring where earthengine doesn't count all the way to numpix
      feat_of_int = image_sampled[j]
      value = feat_of_int['properties']['VV']
      vals.append(value)

  np_vals = np.array(vals)
  my_mean = np_vals.mean()
  my_stdev = np_vals.std()

  val1 = my_mean - my_stdev
  val2 = my_mean - (0.5 * my_stdev)
  val3 = my_mean
  val4 = my_mean + (0.5 * my_stdev)
  val5 = my_mean + (my_stdev)
  return [val1, val2, val3, val4, val5]

In [ ]:
win3_values = np.floor(np.array(value_obtainer(ic_pmedian_3, aoi, 1e6, iceye_scale)))
win5_values = np.floor(np.array(value_obtainer(ic_pmedian_5, aoi, 1e6, iceye_scale)))
win7_values = np.floor(np.array(value_obtainer(ic_pmedian_7, aoi, 1e6, iceye_scale)))
win9_values = np.floor(np.array(value_obtainer(ic_pmedian_9, aoi, 1e6, iceye_scale)))
win11_values = np.floor(np.array(value_obtainer(ic_pmedian_11, aoi, 1e6, iceye_scale)))

# Create indexed values for easier passing to the Edge Otsu Algorithm
win3_negsig = win3_values[0]
win3_neghalfsig = win3_values[1]
win3_mu = win3_values[2]
win3_poshalfsig = win3_values[3]
win3_possig = win3_values[4]

win5_negsig = win5_values[0]
win5_neghalfsig = win5_values[1]
win5_mu = win5_values[2]
win5_poshalfsig = win5_values[3]
win5_possig = win5_values[4]

win7_negsig = win7_values[0]
win7_neghalfsig = win7_values[1]
win7_mu = win7_values[2]
win7_poshalfsig = win7_values[3]
win7_possig = win7_values[4]

win9_negsig = win9_values[0]
win9_neghalfsig = win9_values[1]
win9_mu = win9_values[2]
win9_poshalfsig = win9_values[3]
win9_possig = win9_values[4]

win11_negsig = win11_values[0]
win11_neghalfsig = win11_values[1]
win11_mu = win11_values[2]
win11_poshalfsig = win11_values[3]
win11_possig = win11_values[4]

# Double Check that win3 values are the same as indexed values
print(win3_values)
print(win5_values)
print(win7_values)
print(win9_values)
print(win11_values)

print([win3_negsig, win3_neghalfsig, win3_mu, win3_poshalfsig, win3_possig])
print([win5_negsig, win5_neghalfsig, win5_mu, win5_poshalfsig, win5_possig])
print([win7_negsig, win7_neghalfsig, win7_mu, win7_poshalfsig, win7_possig])
print([win9_negsig, win9_neghalfsig, win9_mu, win9_poshalfsig, win9_possig])
print([win11_negsig, win11_neghalfsig, win11_mu, win11_poshalfsig, win11_possig])

In [ ]:
# Create a histogram for each band
def plot_histograms_w_values(image, bands,  aoi_descriptor, region, xmin, xmax, value_list, final_thresh, scale=3.46325, n_bins=4096):
    """Plot histograms for each band of an ee.Image."""
    fig, axes = plt.subplots(len(bands), 1, figsize=(8, 4*len(bands)))

    if len(bands) == 1:
        axes = [axes]

    for i, band in enumerate(bands):
        # Reduce region to histogram
        hist_dict = image.select(band).reduceRegion(
            reducer=ee.Reducer.histogram(maxBuckets=n_bins), #maxBuckets=n_bins),
            geometry=region,
            scale=scale,

            bestEffort=True
        ).get(band).getInfo()

        if hist_dict is not None:
            counts = hist_dict['histogram']
            bucketMeans = hist_dict['bucketMeans']

            axes[i].bar(bucketMeans, counts, width=(bucketMeans[1]-bucketMeans[0]))
            axes[i].set_title(f"Histogram and thresh_no_data_values for {aoi_descriptor}")
            axes[i].set_xlabel("Pixel values")
            axes[i].set_ylabel("Frequency")
            axes[i].set_xlim(xmin, xmax) #0, 250
        else:
            axes[i].set_title(f"{band}: No data")

    color_list = ['#FF0000', '#00FF00', '#FFFF00', '#00FFFF', '#FF00FF']
    label_list = ['$\mu - \sigma$', '$\mu - 0.5*\sigma$', '$\mu$', '$\mu + 0.5 * \sigma$', '$\mu + \sigma$']

    for j in range(len(value_list)):
      neg2sigma = value_list[0]
      negsigma = value_list[1]
      mu = value_list [2]
      sigma = value_list[3]
      sigma2 = value_list[4]

      #print(thresh_value)
      #print(thresh_label)
      #print(thresh_color)
      plt.axvline(value_list[j], color=color_list[j], label=label_list[j])

    plt.axvline(final_thresh, color='#000000', label = 'Final Threshold')


    plt.legend(loc='upper right')
    #plt.grid()
    plt.tight_layout()
    plt.show()

In [ ]:
plot_histograms_w_values(ic_pmedian_3, ['VV'], 'PMedian Filter, Window of 3 pixels', aoi_geom, 0, 1200, win3_values, 614, scale=iceye_scale, n_bins=1024)

In [ ]:
plot_histograms_w_values(ic_pmedian_5, ['VV'], 'PMedian Filter, Window of 5 pixels', aoi_geom, 0, 1200, win5_values, 582, scale=iceye_scale, n_bins=1024)

In [ ]:
plot_histograms_w_values(ic_pmedian_7, ['VV'], 'PMedian Filter, Window of 7 pixels', aoi_geom, 0, 1200, win7_values, 578, scale=iceye_scale, n_bins=1024)

In [ ]:
plot_histograms_w_values(ic_pmedian_9, ['VV'], 'PMedian Filter, Window of 9 pixels', aoi_geom, 0, 1200, win9_values, 569, scale=iceye_scale, n_bins=1024)

In [ ]:
plot_histograms_w_values(ic_pmedian_11, ['VV'], 'PMedian Filter, Window of 11 pixels', aoi_geom, 0, 1200, win11_values, 561, scale=iceye_scale, n_bins=1024)

# Step 4: Run all parameterizations

since changing the number of pixels used to sample to find the initial threshold does not seem to impact the initial threshold, we will not parameterize this variable. This is because without parameterizing this variable, we will still have 125 different permutations to test.

Recall that the naming convention we will use to keep the permutations straight is thresholdingAlg_speckleWindow_pointsForInitialSampling_InitThreshold_threshNoDataPosition_threshNoDataValue_connectedPixels

* Window of 3
  * Initial threshold using 10,000 pixels
    * thresh_no_data using mu - sigma
      * 50 connected pxiels
        * This branch corresponds to an initial threshold of 580 and a thresh_no_data value of 256
        * Thus, the file name will be called edgeotsu_Pmedian3_1e4_580_negsigma_356_50
      * 100 connected pixels
      * 200 connected pixels
      * 300 connected pixels
      * 350 connected pixels
    * thresh_no_data using mu - sigma/2
      * 50 connected pixels
      * 100 connected pixels
      * 200 connected pixels
      * 300 connected pixels
      * 350 connected pixels
    * thresh_no_data using mu
      * 50 connected pixels
      * 100 connected pixels
      * 200 connected pixels
      * 300 connected pixels
      * 350 connected pixels
    * thresh_no_data using mu + sigma/2
      * 50 connected pixels
      * 100 connected pixels
      * 200 connected pixels
      * 300 connected pixels
      * 350 connected pixels
    * thresh_no_data using mu + sigma
      * 50 connected pixels
      * 100 connected pixels
      * 200 connected pixels
      * 300 connected pixels
      * 350 connected pixels
* Window of 5
  * copy above tree from Window of 3
* window of 7
  * copy above tree from window of 3
* window of 9
  * copy tree from window 3
* window of 11
  * copy tree from window 3

### 4a: PMedian, Window of 3

In [ ]:
# Window of 3, thresh_no_data mu - sigma, 50 connected pixels

edgeotsu_01_0_01_50 = hf.edge_otsu(
    ic_pmedian_3,
    band = 'VV',
    scale = iceye_scale,
    connected_pixels=200,
    initial_threshold=ic_pmedian_3_1e4_threshInit,
    thresh_no_data = win3_negsig,
    region = aoi_geom,
    canny_threshold=0.01,
    canny_sigma=0,
    canny_lt=0.01,
    edge_buffer=50,
    return_threshold = True
)

In [ ]:
def final_thresh(image):
  ft = image.getInfo()['bands'][0]['data_type']['min']
  return ft

# Part 4: Canny Sigma 0

## Part 4a: Canny Threshold 0.01

### Part 4a1: Canny_LT 0.01

In [ ]:
edgeotsu_0_01_01_50 = hf.edge_otsu(
    ic_pmedian_3,
    band = 'VV',
    scale = iceye_scale,
    connected_pixels=200,
    initial_threshold=ic_pmedian_3_1e4_threshInit,
    thresh_no_data = win3_negsig,
    region = aoi_geom,
    canny_threshold=0.01,
    canny_sigma=0,
    canny_lt=0.01,
    edge_buffer=50,
    return_threshold = True
)

In [ ]:
edgeotsu_0_01_01_100 = hf.edge_otsu(
    ic_pmedian_3,
    band = 'VV',
    scale = iceye_scale,
    connected_pixels=200,
    initial_threshold=ic_pmedian_3_1e4_threshInit,
    thresh_no_data = win3_negsig,
    region = aoi_geom,
    canny_threshold=0.01,
    canny_sigma=0,
    canny_lt=0.01,
    edge_buffer=100,
    return_threshold = True
)

In [ ]:
edgeotsu_0_01_01_250 = hf.edge_otsu(
    ic_pmedian_3,
    band = 'VV',
    scale = iceye_scale,
    connected_pixels=200,
    initial_threshold=ic_pmedian_3_1e4_threshInit,
    thresh_no_data = win3_negsig,
    region = aoi_geom,
    canny_threshold=0.01,
    canny_sigma=0,
    canny_lt=0.01,
    edge_buffer=250,
    return_threshold = True
)

In [ ]:
print(final_thresh(edgeotsu_0_01_01_50))


In [ ]:
print(final_thresh(edgeotsu_0_01_01_100))

In [ ]:
print(final_thresh(edgeotsu_0_01_01_250))

### Part 4a2: Canny_LT 0.05

In [ ]:
edgeotsu_0_01_05_50 = hf.edge_otsu(
    ic_pmedian_3,
    band = 'VV',
    scale = iceye_scale,
    connected_pixels=200,
    initial_threshold=ic_pmedian_3_1e4_threshInit,
    thresh_no_data = win3_negsig,
    region = aoi_geom,
    canny_threshold=0.01,
    canny_sigma=0,
    canny_lt=0.05,
    edge_buffer=50,
    return_threshold = True
)

edgeotsu_0_01_05_100 = hf.edge_otsu(
    ic_pmedian_3,
    band = 'VV',
    scale = iceye_scale,
    connected_pixels=200,
    initial_threshold=ic_pmedian_3_1e4_threshInit,
    thresh_no_data = win3_negsig,
    region = aoi_geom,
    canny_threshold=0.01,
    canny_sigma=0,
    canny_lt=0.05,
    edge_buffer=100,
    return_threshold = True
)

edgeotsu_0_01_05_250 = hf.edge_otsu(
    ic_pmedian_3,
    band = 'VV',
    scale = iceye_scale,
    connected_pixels=200,
    initial_threshold=ic_pmedian_3_1e4_threshInit,
    thresh_no_data = win3_negsig,
    region = aoi_geom,
    canny_threshold=0.01,
    canny_sigma=0,
    canny_lt=0.05,
    edge_buffer=250,
    return_threshold = True
)

In [ ]:
print(final_thresh(edgeotsu_0_01_05_50))

In [ ]:
print(final_thresh(edgeotsu_0_01_05_100))

In [ ]:
print(final_thresh(edgeotsu_0_01_05_250))

### Part 4a3: Canny_LT 0.1

In [ ]:
edgeotsu_0_01_1_50 = hf.edge_otsu(
    ic_pmedian_3,
    band = 'VV',
    scale = iceye_scale,
    connected_pixels=200,
    initial_threshold=ic_pmedian_3_1e4_threshInit,
    thresh_no_data = win3_negsig,
    region = aoi_geom,
    canny_threshold=0.01,
    canny_sigma=0,
    canny_lt=0.1,
    edge_buffer=50,
    return_threshold = True
)

edgeotsu_0_01_1_100 = hf.edge_otsu(
    ic_pmedian_3,
    band = 'VV',
    scale = iceye_scale,
    connected_pixels=200,
    initial_threshold=ic_pmedian_3_1e4_threshInit,
    thresh_no_data = win3_negsig,
    region = aoi_geom,
    canny_threshold=0.01,
    canny_sigma=0,
    canny_lt=0.1,
    edge_buffer=100,
    return_threshold = True
)

edgeotsu_0_01_1_250 = hf.edge_otsu(
    ic_pmedian_3,
    band = 'VV',
    scale = iceye_scale,
    connected_pixels=200,
    initial_threshold=ic_pmedian_3_1e4_threshInit,
    thresh_no_data = win3_negsig,
    region = aoi_geom,
    canny_threshold=0.01,
    canny_sigma=0,
    canny_lt=0.1,
    edge_buffer=250,
    return_threshold = True
)

In [ ]:
print(final_thresh(edgeotsu_0_01_1_50))

In [ ]:
print(final_thresh(edgeotsu_0_01_1_100))

In [ ]:
print(final_thresh(edgeotsu_0_01_1_250))

## Part 4b: Canny Threshold 0.05

### Part 4b1: Canny_LT 0.01

In [ ]:
edgeotsu_0_05_01_50 = hf.edge_otsu(
    ic_pmedian_3,
    band = 'VV',
    scale = iceye_scale,
    connected_pixels=200,
    initial_threshold=ic_pmedian_3_1e4_threshInit,
    thresh_no_data = win3_negsig,
    region = aoi_geom,
    canny_threshold=0.05,
    canny_sigma=0,
    canny_lt=0.01,
    edge_buffer=50,
    return_threshold = True
)

edgeotsu_0_05_01_100 = hf.edge_otsu(
    ic_pmedian_3,
    band = 'VV',
    scale = iceye_scale,
    connected_pixels=200,
    initial_threshold=ic_pmedian_3_1e4_threshInit,
    thresh_no_data = win3_negsig,
    region = aoi_geom,
    canny_threshold=0.05,
    canny_sigma=0,
    canny_lt=0.01,
    edge_buffer=100,
    return_threshold = True
)

edgeotsu_0_05_01_250 = hf.edge_otsu(
    ic_pmedian_3,
    band = 'VV',
    scale = iceye_scale,
    connected_pixels=200,
    initial_threshold=ic_pmedian_3_1e4_threshInit,
    thresh_no_data = win3_negsig,
    region = aoi_geom,
    canny_threshold=0.05,
    canny_sigma=0,
    canny_lt=0.01,
    edge_buffer=250,
    return_threshold = True
)

In [ ]:
print(final_thresh(edgeotsu_0_05_01_50))

In [ ]:
print(final_thresh(edgeotsu_0_05_01_100))

In [ ]:
print(final_thresh(edgeotsu_0_05_01_250))

### Part 4b2: Canny_LT 0.05

In [ ]:
edgeotsu_0_05_05_50 = hf.edge_otsu(
    ic_pmedian_3,
    band = 'VV',
    scale = iceye_scale,
    connected_pixels=200,
    initial_threshold=ic_pmedian_3_1e4_threshInit,
    thresh_no_data = win3_negsig,
    region = aoi_geom,
    canny_threshold=0.05,
    canny_sigma=0,
    canny_lt=0.05,
    edge_buffer=50,
    return_threshold = True
)

edgeotsu_0_05_05_100 = hf.edge_otsu(
    ic_pmedian_3,
    band = 'VV',
    scale = iceye_scale,
    connected_pixels=200,
    initial_threshold=ic_pmedian_3_1e4_threshInit,
    thresh_no_data = win3_negsig,
    region = aoi_geom,
    canny_threshold=0.05,
    canny_sigma=0,
    canny_lt=0.05,
    edge_buffer=100,
    return_threshold = True
)

edgeotsu_0_05_05_250 = hf.edge_otsu(
    ic_pmedian_3,
    band = 'VV',
    scale = iceye_scale,
    connected_pixels=200,
    initial_threshold=ic_pmedian_3_1e4_threshInit,
    thresh_no_data = win3_negsig,
    region = aoi_geom,
    canny_threshold=0.05,
    canny_sigma=0,
    canny_lt=0.05,
    edge_buffer=250,
    return_threshold = True
)

In [ ]:
print(final_thresh(edgeotsu_0_05_05_50))

In [ ]:
print(final_thresh(edgeotsu_0_05_05_100))

In [ ]:
print(final_thresh(edgeotsu_0_05_05_250))

### Part 4b3: Canny_LT 0.1

In [ ]:
edgeotsu_0_05_1_50 = hf.edge_otsu(
    ic_pmedian_3,
    band = 'VV',
    scale = iceye_scale,
    connected_pixels=200,
    initial_threshold=ic_pmedian_3_1e4_threshInit,
    thresh_no_data = win3_negsig,
    region = aoi_geom,
    canny_threshold=0.05,
    canny_sigma=0,
    canny_lt=0.05,
    edge_buffer=50,
    return_threshold = True
)

edgeotsu_0_05_1_100 = hf.edge_otsu(
    ic_pmedian_3,
    band = 'VV',
    scale = iceye_scale,
    connected_pixels=200,
    initial_threshold=ic_pmedian_3_1e4_threshInit,
    thresh_no_data = win3_negsig,
    region = aoi_geom,
    canny_threshold=0.05,
    canny_sigma=0,
    canny_lt=0.05,
    edge_buffer=100,
    return_threshold = True
)

edgeotsu_0_05_1_250 = hf.edge_otsu(
    ic_pmedian_3,
    band = 'VV',
    scale = iceye_scale,
    connected_pixels=200,
    initial_threshold=ic_pmedian_3_1e4_threshInit,
    thresh_no_data = win3_negsig,
    region = aoi_geom,
    canny_threshold=0.05,
    canny_sigma=0,
    canny_lt=0.05,
    edge_buffer=250,
    return_threshold = True
)

In [ ]:
print(final_thresh(edgeotsu_0_05_1_50))

In [ ]:
print(final_thresh(edgeotsu_0_05_1_100))

In [ ]:
print(final_thresh(edgeotsu_0_05_1_250))

## Part 4c: Canny Threshold 0.1

### Part 4c1: Canny_LT 0.01

In [ ]:
edgeotsu_0_1_01_50 = hf.edge_otsu(
    ic_pmedian_3,
    band = 'VV',
    scale = iceye_scale,
    connected_pixels=200,
    initial_threshold=ic_pmedian_3_1e4_threshInit,
    thresh_no_data = win3_negsig,
    region = aoi_geom,
    canny_threshold=0.1,
    canny_sigma=0,
    canny_lt=0.01,
    edge_buffer=50,
    return_threshold = True
)

edgeotsu_0_1_01_100 = hf.edge_otsu(
    ic_pmedian_3,
    band = 'VV',
    scale = iceye_scale,
    connected_pixels=200,
    initial_threshold=ic_pmedian_3_1e4_threshInit,
    thresh_no_data = win3_negsig,
    region = aoi_geom,
    canny_threshold=0.1,
    canny_sigma=0,
    canny_lt=0.01,
    edge_buffer=100,
    return_threshold = True
)

edgeotsu_0_1_01_250 = hf.edge_otsu(
    ic_pmedian_3,
    band = 'VV',
    scale = iceye_scale,
    connected_pixels=200,
    initial_threshold=ic_pmedian_3_1e4_threshInit,
    thresh_no_data = win3_negsig,
    region = aoi_geom,
    canny_threshold=0.1,
    canny_sigma=0,
    canny_lt=0.01,
    edge_buffer=250,
    return_threshold = True
)

In [ ]:
final_thresh(edgeotsu_0_1_01_50)

In [ ]:
final_thresh(edgeotsu_0_1_01_100)

In [ ]:
final_thresh(edgeotsu_0_1_01_250)

### Part 4c2: Canny_LT 0.05

In [ ]:
edgeotsu_0_1_05_50 = hf.edge_otsu(
    ic_pmedian_3,
    band = 'VV',
    scale = iceye_scale,
    connected_pixels=200,
    initial_threshold=ic_pmedian_3_1e4_threshInit,
    thresh_no_data = win3_negsig,
    region = aoi_geom,
    canny_threshold=0.1,
    canny_sigma=0,
    canny_lt=0.05,
    edge_buffer=50,
    return_threshold = True
)

edgeotsu_0_1_05_100 = hf.edge_otsu(
    ic_pmedian_3,
    band = 'VV',
    scale = iceye_scale,
    connected_pixels=200,
    initial_threshold=ic_pmedian_3_1e4_threshInit,
    thresh_no_data = win3_negsig,
    region = aoi_geom,
    canny_threshold=0.1,
    canny_sigma=0,
    canny_lt=0.05,
    edge_buffer=100,
    return_threshold = True
)

edgeotsu_0_1_05_250 = hf.edge_otsu(
    ic_pmedian_3,
    band = 'VV',
    scale = iceye_scale,
    connected_pixels=200,
    initial_threshold=ic_pmedian_3_1e4_threshInit,
    thresh_no_data = win3_negsig,
    region = aoi_geom,
    canny_threshold=0.1,
    canny_sigma=0,
    canny_lt=0.05,
    edge_buffer=250,
    return_threshold = True
)

In [ ]:
final_thresh(edgeotsu_0_1_05_50)

In [ ]:
final_thresh(edgeotsu_0_1_05_100)

In [ ]:
final_thresh(edgeotsu_0_1_05_250)

### Part 4c3: Canny_LT 0.1

In [ ]:
edgeotsu_0_1_1_50 = hf.edge_otsu(
    ic_pmedian_3,
    band = 'VV',
    scale = iceye_scale,
    connected_pixels=200,
    initial_threshold=ic_pmedian_3_1e4_threshInit,
    thresh_no_data = win3_negsig,
    region = aoi_geom,
    canny_threshold=0.1,
    canny_sigma=0,
    canny_lt=0.1,
    edge_buffer=50,
    return_threshold = True
)

edgeotsu_0_1_1_100 = hf.edge_otsu(
    ic_pmedian_3,
    band = 'VV',
    scale = iceye_scale,
    connected_pixels=200,
    initial_threshold=ic_pmedian_3_1e4_threshInit,
    thresh_no_data = win3_negsig,
    region = aoi_geom,
    canny_threshold=0.1,
    canny_sigma=0,
    canny_lt=0.1,
    edge_buffer=100,
    return_threshold = True
)

edgeotsu_0_1_1_250 = hf.edge_otsu(
    ic_pmedian_3,
    band = 'VV',
    scale = iceye_scale,
    connected_pixels=200,
    initial_threshold=ic_pmedian_3_1e4_threshInit,
    thresh_no_data = win3_negsig,
    region = aoi_geom,
    canny_threshold=0.1,
    canny_sigma=0,
    canny_lt=0.1,
    edge_buffer=250,
    return_threshold = True
)

In [ ]:
final_thresh(edgeotsu_0_1_1_50)

In [ ]:
final_thresh(edgeotsu_0_1_1_100)

In [ ]:
final_thresh(edgeotsu_0_1_1_250)

# Part 5: Canny Sigma -1

## Part 5a: Canny Threshold 0.01

### Part 5a1: Canny_LT 0.01

In [ ]:
edgeotsu_neg1_1_01_50 = hf.edge_otsu(
    ic_pmedian_3,
    band = 'VV',
    scale = iceye_scale,
    connected_pixels=200,
    initial_threshold=ic_pmedian_3_1e4_threshInit,
    thresh_no_data = win3_negsig,
    region = aoi_geom,
    canny_threshold=0.1,
    canny_sigma=-1,
    canny_lt=0.01,
    edge_buffer=50,
    return_threshold = True
)

edgeotsu_neg1_1_01_100 = hf.edge_otsu(
    ic_pmedian_3,
    band = 'VV',
    scale = iceye_scale,
    connected_pixels=200,
    initial_threshold=ic_pmedian_3_1e4_threshInit,
    thresh_no_data = win3_negsig,
    region = aoi_geom,
    canny_threshold=0.1,
    canny_sigma=-1,
    canny_lt=0.01,
    edge_buffer=100,
    return_threshold = True
)

edgeotsu_neg1_1_01_250 = hf.edge_otsu(
    ic_pmedian_3,
    band = 'VV',
    scale = iceye_scale,
    connected_pixels=200,
    initial_threshold=ic_pmedian_3_1e4_threshInit,
    thresh_no_data = win3_negsig,
    region = aoi_geom,
    canny_threshold=0.1,
    canny_sigma=-1,
    canny_lt=0.01,
    edge_buffer=250,
    return_threshold = True
)

In [ ]:
final_thresh(edgeotsu_neg1_1_01_50)

In [ ]:
final_thresh(edgeotsu_neg1_1_01_100)

In [ ]:
final_thresh(edgeotsu_neg1_1_01_250)

### Part 5a2: Canny_LT 0.05

In [ ]:
edgeotsu_neg1_1_05_50 = hf.edge_otsu(
    ic_pmedian_3,
    band = 'VV',
    scale = iceye_scale,
    connected_pixels=200,
    initial_threshold=ic_pmedian_3_1e4_threshInit,
    thresh_no_data = win3_negsig,
    region = aoi_geom,
    canny_threshold=0.1,
    canny_sigma=-1,
    canny_lt=0.05,
    edge_buffer=50,
    return_threshold = True
)

edgeotsu_neg1_1_05_100 = hf.edge_otsu(
    ic_pmedian_3,
    band = 'VV',
    scale = iceye_scale,
    connected_pixels=200,
    initial_threshold=ic_pmedian_3_1e4_threshInit,
    thresh_no_data = win3_negsig,
    region = aoi_geom,
    canny_threshold=0.1,
    canny_sigma=-1,
    canny_lt=0.05,
    edge_buffer=100,
    return_threshold = True
)

edgeotsu_neg1_1_05_250 = hf.edge_otsu(
    ic_pmedian_3,
    band = 'VV',
    scale = iceye_scale,
    connected_pixels=200,
    initial_threshold=ic_pmedian_3_1e4_threshInit,
    thresh_no_data = win3_negsig,
    region = aoi_geom,
    canny_threshold=0.1,
    canny_sigma=-1,
    canny_lt=0.05,
    edge_buffer=250,
    return_threshold = True
)

In [ ]:
final_thresh(edgeotsu_neg1_1_05_50)

In [ ]:
final_thresh(edgeotsu_neg1_1_05_100)

In [ ]:
final_thresh(edgeotsu_neg1_1_05_250)

### Part 5a3: Canny_LT 0.1

In [ ]:
edgeotsu_neg1_1_1_50 = hf.edge_otsu(
    ic_pmedian_3,
    band = 'VV',
    scale = iceye_scale,
    connected_pixels=200,
    initial_threshold=ic_pmedian_3_1e4_threshInit,
    thresh_no_data = win3_negsig,
    region = aoi_geom,
    canny_threshold=0.1,
    canny_sigma=-1,
    canny_lt=0.1,
    edge_buffer=50,
    return_threshold = True
)

edgeotsu_neg1_1_1_100 = hf.edge_otsu(
    ic_pmedian_3,
    band = 'VV',
    scale = iceye_scale,
    connected_pixels=200,
    initial_threshold=ic_pmedian_3_1e4_threshInit,
    thresh_no_data = win3_negsig,
    region = aoi_geom,
    canny_threshold=0.1,
    canny_sigma=-1,
    canny_lt=0.1,
    edge_buffer=100,
    return_threshold = True
)

edgeotsu_neg1_1_1_250 = hf.edge_otsu(
    ic_pmedian_3,
    band = 'VV',
    scale = iceye_scale,
    connected_pixels=200,
    initial_threshold=ic_pmedian_3_1e4_threshInit,
    thresh_no_data = win3_negsig,
    region = aoi_geom,
    canny_threshold=0.1,
    canny_sigma=-1,
    canny_lt=1,
    edge_buffer=250,
    return_threshold = True
)

In [ ]:
final_thresh(edgeotsu_neg1_1_1_50)

In [ ]:
final_thresh(edgeotsu_neg1_1_1_100)

In [ ]:
final_thresh(edgeotsu_neg1_1_1_250)

## Part 5b: Canny Threshold 0.05

### Part 5b1: Canny_LT 0.01

In [ ]:
edgeotsu_neg1_05_01_50 = hf.edge_otsu(
    ic_pmedian_3,
    band = 'VV',
    scale = iceye_scale,
    connected_pixels=200,
    initial_threshold=ic_pmedian_3_1e4_threshInit,
    thresh_no_data = win3_negsig,
    region = aoi_geom,
    canny_threshold=0.05,
    canny_sigma=-1,
    canny_lt=0.01,
    edge_buffer=50,
    return_threshold = True
)

edgeotsu_neg1_05_01_100 = hf.edge_otsu(
    ic_pmedian_3,
    band = 'VV',
    scale = iceye_scale,
    connected_pixels=200,
    initial_threshold=ic_pmedian_3_1e4_threshInit,
    thresh_no_data = win3_negsig,
    region = aoi_geom,
    canny_threshold=0.05,
    canny_sigma=-1,
    canny_lt=0.01,
    edge_buffer=100,
    return_threshold = True
)

edgeotsu_neg1_05_01_250 = hf.edge_otsu(
    ic_pmedian_3,
    band = 'VV',
    scale = iceye_scale,
    connected_pixels=200,
    initial_threshold=ic_pmedian_3_1e4_threshInit,
    thresh_no_data = win3_negsig,
    region = aoi_geom,
    canny_threshold=0.05,
    canny_sigma=-1,
    canny_lt=0.01,
    edge_buffer=250,
    return_threshold = True
)

In [ ]:
final_thresh(edgeotsu_neg1_05_01_50)

In [ ]:
final_thresh(edgeotsu_neg1_05_01_100)

In [ ]:
final_thresh(edgeotsu_neg1_05_01_250)

### Part 5b2: Canny_LT 0.05

In [ ]:
edgeotsu_neg1_05_05_50 = hf.edge_otsu(
    ic_pmedian_3,
    band = 'VV',
    scale = iceye_scale,
    connected_pixels=200,
    initial_threshold=ic_pmedian_3_1e4_threshInit,
    thresh_no_data = win3_negsig,
    region = aoi_geom,
    canny_threshold=0.05,
    canny_sigma=-1,
    canny_lt=0.05,
    edge_buffer=50,
    return_threshold = True
)

edgeotsu_neg1_05_05_100 = hf.edge_otsu(
    ic_pmedian_3,
    band = 'VV',
    scale = iceye_scale,
    connected_pixels=200,
    initial_threshold=ic_pmedian_3_1e4_threshInit,
    thresh_no_data = win3_negsig,
    region = aoi_geom,
    canny_threshold=0.05,
    canny_sigma=-1,
    canny_lt=0.05,
    edge_buffer=100,
    return_threshold = True
)

edgeotsu_neg1_05_05_250 = hf.edge_otsu(
    ic_pmedian_3,
    band = 'VV',
    scale = iceye_scale,
    connected_pixels=200,
    initial_threshold=ic_pmedian_3_1e4_threshInit,
    thresh_no_data = win3_negsig,
    region = aoi_geom,
    canny_threshold=0.05,
    canny_sigma=-1,
    canny_lt=0.05,
    edge_buffer=250,
    return_threshold = True
)

In [ ]:
final_thresh(edgeotsu_neg1_05_05_50)

In [ ]:
final_thresh(edgeotsu_neg1_05_05_100)

In [ ]:
final_thresh(edgeotsu_neg1_05_05_250)

### Part 5b3: Canny_LT 0.1

In [ ]:
edgeotsu_neg1_05_1_50 = hf.edge_otsu(
    ic_pmedian_3,
    band = 'VV',
    scale = iceye_scale,
    connected_pixels=200,
    initial_threshold=ic_pmedian_3_1e4_threshInit,
    thresh_no_data = win3_negsig,
    region = aoi_geom,
    canny_threshold=0.05,
    canny_sigma=-1,
    canny_lt=0.1,
    edge_buffer=50,
    return_threshold = True
)

edgeotsu_neg1_05_1_100 = hf.edge_otsu(
    ic_pmedian_3,
    band = 'VV',
    scale = iceye_scale,
    connected_pixels=200,
    initial_threshold=ic_pmedian_3_1e4_threshInit,
    thresh_no_data = win3_negsig,
    region = aoi_geom,
    canny_threshold=0.05,
    canny_sigma=-1,
    canny_lt=0.1,
    edge_buffer=100,
    return_threshold = True
)

edgeotsu_neg1_05_1_250 = hf.edge_otsu(
    ic_pmedian_3,
    band = 'VV',
    scale = iceye_scale,
    connected_pixels=200,
    initial_threshold=ic_pmedian_3_1e4_threshInit,
    thresh_no_data = win3_negsig,
    region = aoi_geom,
    canny_threshold=0.05,
    canny_sigma=-1,
    canny_lt=0.1,
    edge_buffer=250,
    return_threshold = True
)

In [ ]:
final_thresh(edgeotsu_neg1_05_1_50)

In [ ]:
final_thresh(edgeotsu_neg1_05_1_100)

In [ ]:
final_thresh(edgeotsu_neg1_05_1_250)

## Part 5c: Canny Threshold 0.1

In [ ]:
edgeotsu_neg1_1_01_50 = hf.edge_otsu(
    ic_pmedian_3,
    band = 'VV',
    scale = iceye_scale,
    connected_pixels=200,
    initial_threshold=ic_pmedian_3_1e4_threshInit,
    thresh_no_data = win3_negsig,
    region = aoi_geom,
    canny_threshold=0.1,
    canny_sigma=-1,
    canny_lt=0.01,
    edge_buffer=50,
    return_threshold = True
)

edgeotsu_neg1_1_01_100 = hf.edge_otsu(
    ic_pmedian_3,
    band = 'VV',
    scale = iceye_scale,
    connected_pixels=200,
    initial_threshold=ic_pmedian_3_1e4_threshInit,
    thresh_no_data = win3_negsig,
    region = aoi_geom,
    canny_threshold=0.05,
    canny_sigma=-1,
    canny_lt=0.01,
    edge_buffer=100,
    return_threshold = True
)

edgeotsu_neg1_1_01_250 = hf.edge_otsu(
    ic_pmedian_3,
    band = 'VV',
    scale = iceye_scale,
    connected_pixels=200,
    initial_threshold=ic_pmedian_3_1e4_threshInit,
    thresh_no_data = win3_negsig,
    region = aoi_geom,
    canny_threshold=0.1,
    canny_sigma=-1,
    canny_lt=0.01,
    edge_buffer=250,
    return_threshold = True
)

### Part 5c1: Canny_LT 0.01

In [ ]:
final_thresh(edgeotsu_neg1_1_01_50)

In [ ]:
final_thresh(edgeotsu_neg1_1_01_100)

In [ ]:
final_thresh(edgeotsu_neg1_1_01_250)

### Part 5c2: Canny_LT 0.05

### Part 5c3: Canny_LT 0.1

# Part 6: Canny Sigma 1

## Part 6a: Canny Threshold 0.01

### Part 6a1: Canny_LT 0.01

In [ ]:
edgeotsu_1_1_01_50 = hf.edge_otsu(
    ic_pmedian_3,
    band = 'VV',
    scale = iceye_scale,
    connected_pixels=200,
    initial_threshold=ic_pmedian_3_1e4_threshInit,
    thresh_no_data = win3_negsig,
    region = aoi_geom,
    canny_threshold=0.1,
    canny_sigma=1,
    canny_lt=0.01,
    edge_buffer=50,
    return_threshold = True
)

edgeotsu_neg1_1_01_100 = hf.edge_otsu(
    ic_pmedian_3,
    band = 'VV',
    scale = iceye_scale,
    connected_pixels=200,
    initial_threshold=ic_pmedian_3_1e4_threshInit,
    thresh_no_data = win3_negsig,
    region = aoi_geom,
    canny_threshold=0.1,
    canny_sigma=-1,
    canny_lt=0.01,
    edge_buffer=100,
    return_threshold = True
)

edgeotsu_neg1_1_01_250 = hf.edge_otsu(
    ic_pmedian_3,
    band = 'VV',
    scale = iceye_scale,
    connected_pixels=200,
    initial_threshold=ic_pmedian_3_1e4_threshInit,
    thresh_no_data = win3_negsig,
    region = aoi_geom,
    canny_threshold=0.1,
    canny_sigma=-1,
    canny_lt=0.01,
    edge_buffer=250,
    return_threshold = True
)

In [ ]:
final_thresh(edgeotsu_neg1_1_01_50)

In [ ]:
final_thresh(edgeotsu_neg1_1_01_100)

In [ ]:
final_thresh(edgeotsu_neg1_1_01_250)

### Part 6a2: Canny_LT 0.05

### Part 6a3: Canny_LT 0.1

## Part 6b: Canny Threshold 0.05

### Part 6b1: Canny_LT 0.01

### Part 6b2: Canny_LT 0.05

### Part 6b3: Canny_LT 0.1

## Part 6c: Canny Threshold 0.1

### Part 6c1: Canny_LT 0.01

### Part 6c2: Canny_LT 0.05

### Part 6c3: Canny_LT 0.1

### Part 4c:

In [ ]:
print(edgeotsu_01_0_01_50.getInfo())